# RAG (Web Search) — `career_position` Annotation — Broad sectors

For each career entry, performs two Google searches (workplace + job title),
synthesises the top-3 snippets into a short context description, then feeds
that context into the broad sector classification prompt.

**Model:** TBD — set `MODEL_ID` below  
**Requires:** `OPENAI_API_KEY` (or LM Studio), `GOOGLE_API_KEY`, `GOOGLE_CSE_ID` in `.env`

## 1. Imports & config

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import anthropic
import json
import os
import time
from pathlib import Path

import pandas as pd
import requests
from openai import OpenAI
from tqdm import tqdm

from corex_eval import evaluate, load_inputs, CAREER_POSITION_SECTORS, CAREER_POSITION_SECTOR_HINTS

# ── Classification model (Anthropic Claude) ──────────────────────────────────
CLASSIFY_MODEL_ID = "claude-opus-4-6"
TEMPERATURE       = 0
MAX_TOKENS        = 16

# ── Synthesis model (LM Studio — GPT-OSS 20B) ───────────────────────────────
SYNTH_MODEL_ID    = "gpt-oss-20b"
LMSTUDIO_URL      = "http://localhost:1234/v1"
SYNTH_MAX_TOKENS  = 5000

# ── Google Custom Search ─────────────────────────────────────────────────────
GOOGLE_API_KEY   = os.environ["GOOGLE_API_KEY"]
GOOGLE_CSE_ID    = os.environ["GOOGLE_CSE_ID"]
N_SEARCH_RESULTS = 3

# ── Cache ────────────────────────────────────────────────────────────────────
CACHE_DIR            = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
SEARCH_CACHE_FILE    = CACHE_DIR / "search_cache.json"
SYNTHESIS_CACHE_FILE = CACHE_DIR / "synthesis_cache.json"

classify_client = anthropic.Anthropic()
synth_client    = OpenAI(base_url=LMSTUDIO_URL, api_key="lm-studio")
print(f"Classification: {CLASSIFY_MODEL_ID}  |  Synthesis: {SYNTH_MODEL_ID}")


Classification: claude-opus-4-6  |  Synthesis: gpt-oss-20b


## 2. Cache helpers

In [3]:
def _load_cache(path):
    return json.load(open(path)) if path.exists() else {}

def _save_cache(cache, path):
    json.dump(cache, open(path, "w"))

search_cache    = _load_cache(SEARCH_CACHE_FILE)
synthesis_cache = _load_cache(SYNTHESIS_CACHE_FILE)
print(f"Loaded {len(search_cache)} cached searches, {len(synthesis_cache)} cached syntheses")

Loaded 74 cached searches, 73 cached syntheses


## 3. Build system prompt

In [4]:
sectors_text = "\n".join(
    f"  {k} = {v}: {CAREER_POSITION_SECTOR_HINTS[k]}"
    for k, v in sorted(CAREER_POSITION_SECTORS.items())
)

CLASSIFICATION_SYSTEM_PROMPT = (
    "You are an expert political scientist coding political career histories "
    "using the CoREx codebook.\n\n"
    "Task: given a career entry (job title, workplace, and optional context), "
    "assign the single most appropriate broad sector code.\n"
    'Respond with ONLY the sector number (e.g. "1") — nothing else.\n\n'
    f"Broad sector codes:\n{sectors_text}"
)

SYNTHESIS_SYSTEM_PROMPT = (
    "You are a concise research assistant. Based on the provided web search snippets, "
    "write 1-2 sentences describing the subject. Focus on its sector, role, or function. "
    "If the snippets are not relevant or insufficient, respond with exactly: NA"
)

valid_codes = [str(k) for k in sorted(CAREER_POSITION_SECTORS.keys(), key=int)]
print(f"Broad sectors: {valid_codes}")

Broad sectors: ['1', '2', '3', '4', '5', '6', '7', '8', '9']


## 4. Load test inputs

In [5]:
test_df = load_inputs(task="annotation", variable="career_position")
print(f"{len(test_df)} test rows")
test_df[["case_id", "spell_index", "job_description_label", "workplace_label"]].head()

,case_id,spell_index,job_description_label,workplace_label
0,002002,1,Minister of Infrastructure and Energy,Government of Albania
1,002002,2,General Director,"ALBCONTROL, JSC"
2,002002,3,Public Relations Advisor and Director of the M...,Municipality of Tirana
3,002015,1,Minister of Interior,Government of Albania
4,002015,3,Secretary General,Socialist Party


## 5. Web search & synthesis (cached)

In [6]:
SYNTHESIS_SYSTEM_PROMPT = (
    "Summarize web search snippets in one sentence. "
    "If the snippets are off-topic or insufficient, reply with exactly: NA"
)


def _synth_user_msg(snippets, *, workplace="", job_title=""):
    snippets_text = "\n".join(f"[{i+1}] {s}" for i, s in enumerate(snippets))
    if job_title:
        return (
            f"Web search results for the role '{job_title}' at '{workplace}':\n"
            f"{snippets_text}\n\n"
            f"Summarize in 1 sentence what this role entails."
        )
    else:
        return (
            f"Web search results for the organization '{workplace}':\n"
            f"{snippets_text}\n\n"
            f"Summarize in 1 sentence what this organization is and what sector it belongs to."
        )


def google_search(query):
    if query in search_cache:
        return search_cache[query]
    try:
        resp = requests.get(
            "https://www.googleapis.com/customsearch/v1",
            params={"q": query, "key": GOOGLE_API_KEY, "cx": GOOGLE_CSE_ID, "num": N_SEARCH_RESULTS},
            timeout=10,
        )
        snippets = [item.get("snippet", "") for item in resp.json().get("items", [])]
    except Exception as e:
        print(f"Search error '{query}': {e}")
        snippets = []
    search_cache[query] = snippets
    _save_cache(search_cache, SEARCH_CACHE_FILE)
    time.sleep(0.1)
    return snippets


def synthesise(snippets, *, workplace="", job_title=""):
    """Summarise search snippets for a workplace or a job role via LM Studio."""
    cache_key = f"{job_title}||{workplace}"
    if cache_key in synthesis_cache:
        return synthesis_cache[cache_key]
    if not snippets:
        synthesis_cache[cache_key] = "NA"
        _save_cache(synthesis_cache, SYNTHESIS_CACHE_FILE)
        return "NA"
    try:
        resp = synth_client.chat.completions.create(
            model=SYNTH_MODEL_ID,
            messages=[
                {"role": "system", "content": SYNTHESIS_SYSTEM_PROMPT},
                {"role": "user",   "content": _synth_user_msg(snippets, workplace=workplace, job_title=job_title)},
            ],
            temperature=0,
            max_tokens=SYNTH_MAX_TOKENS,
        )
        result = resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"Synthesis error: {e}")
        result = "NA"
    synthesis_cache[cache_key] = result
    _save_cache(synthesis_cache, SYNTHESIS_CACHE_FILE)
    return result


## 6. Inference

In [7]:
def parse_prediction(text, valid_codes):
    text = text.strip().strip("\"'")
    if text in valid_codes:
        return text
    for code in valid_codes:
        if text.startswith(code):
            return code
    return text


predictions = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    job_title = str(row["job_description_label"] or "")
    workplace = str(row.get("workplace_label") or "")

    country      = str(row.get("country_label") or "")
    wp_snippets  = google_search(f"{workplace} {country} organization")
    job_snippets = google_search(f"{job_title} {workplace} {country}")
    wp_desc      = synthesise(wp_snippets,  workplace=workplace)
    role_desc    = synthesise(job_snippets, workplace=workplace, job_title=job_title)

    context_block = ""
    if wp_desc   and wp_desc   != "NA":
        context_block += f"\nAbout the workplace: {wp_desc}"
    if role_desc and role_desc != "NA":
        context_block += f"\nAbout the role: {role_desc}"

    print(context_block)

    user_msg = f"Job title: {job_title}\nWorkplace: {workplace or 'unknown'}{context_block}"

    try:
        resp = classify_client.messages.create(
            model=CLASSIFY_MODEL_ID,
            system=CLASSIFICATION_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_msg}],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
        )
        raw = resp.content[0].text or ""
    except Exception as e:
        print(f"Error on {row['case_id']}/{row['spell_index']}: {e}")
        raw = ""

    predictions.append(parse_prediction(raw, valid_codes))

predictions_df = test_df[["case_id", "spell_index"]].copy()
predictions_df["predicted_code"] = predictions
predictions_df.head()


,case_id,spell_index,predicted_code
0,002002,1,1
1,002002,2,3
2,002002,3,4
3,002015,1,1
4,002015,3,4


## 7. Evaluate

In [13]:
results = evaluate(
    predictions_df,
    task="annotation",
    variable="career_position",
    granularity="broad",
    submit=True,
    experiment_path="/home/tom/projects/corex_eval/experiments/annotation/rag_websearch_broad/config.yaml"
)
pd.Series({k: v for k, v in results.items() if k not in ("per_class", "per_country")})

accuracy                      0.652969
macro_f1                      0.513965
weighted_f1                   0.650948
semantic_similarity               None
n_classes                            9
n_evaluated                       3553
n_skipped                            0
granularity                      broad
task                        annotation
variable               career_position
dtype: object